# 响应式页面实践

学习目标：能为内容、目录和表单共用一份响应式样式，并按尺寸、内容与交互状态检查页面。

前置知识：HTML 语义结构、资源路径、Flexbox、Grid、媒体查询、容器查询和自定义属性。

适用范围：面向现代浏览器的页面整合实践；容器尺寸查询按支持情况增强。本章不使用 JavaScript，不连接后台。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/22-responsive-page-practice/。

1. [index.html](scripts/22-responsive-page-practice/index.html)：内容页与长文本、图片。
2. [catalog.html](scripts/22-responsive-page-practice/catalog.html)：目录卡片。
3. [form.html](scripts/22-responsive-page-practice/form.html)：原生表单与校验。
4. [styles.css](scripts/22-responsive-page-practice/styles.css)：三页共用的布局、主题和状态。
5. [layout.svg](scripts/22-responsive-page-practice/layout.svg)：本地自制布局示意图。

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/css
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8101 --bind 127.0.0.1
```

Step 3：打开[本章示例首页](http://127.0.0.1:8101/scripts/22-responsive-page-practice/index.html)。

保存修改后刷新页面。

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 三种页面共用基础样式

内容页负责阅读，目录页负责导航，表单页负责输入。三页保留各自的 HTML 结构；本章把颜色、间距和交互外观统一到一份外部样式表。

&lt;meta&gt; 的 viewport 设置让移动浏览器按设备宽度建立布局视口；不添加限制用户缩放的参数。&lt;link&gt; 的 styles.css 相对于当前 HTML 文件，三个页面都位于同一目录。

自定义属性 --paper、--ink、--link、--line 和 --error 分别表示背景、正文、链接、边界和错误颜色；它们是本例约定的名字。1rem 对应根元素字号，让字号和间距随根字号变化。

```html
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>阅读与记录 · 内容页</title>
<link rel="stylesheet" href="styles.css">
```

```css
:root {
  --paper: #ffffff;
  --ink: #17212b;
  --link: #0645ad;
  --line: #596573;
  --error: #a01818;
  color-scheme: light;
}
```

配套文件：[index.html](scripts/22-responsive-page-practice/index.html)、[styles.css](scripts/22-responsive-page-practice/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/22-responsive-page-practice/index.html)

## 2 本章使用的主要属性

| 完整属性名 | 中文名称／含义 | 用途或对象 |
| --- | --- | --- |
| display | 布局方式 | 页面网格、导航和卡片 |
| flex-wrap | 弹性换行 | 允许导航在窄屏折行 |
| gap | 项目间距 | 导航、网格和卡片 |
| grid-template-columns | 网格列轨道 | 页面和组件的列宽 |
| min-inline-size | 行内方向最小尺寸 | 解除项目自动最小尺寸限制 |
| max-inline-size | 行内方向最大尺寸 | 限制正文行宽和图片宽度 |
| block-size | 块方向尺寸 | 图片按比例自动取高 |
| container-type | 查询容器类型 | 建立行内尺寸查询条件 |
| container-name | 查询容器名称 | 选择卡片外层容器 |
| overflow-wrap | 溢出断行策略 | 处理连续长标识 |
| color-scheme | 支持的配色方案 | 原生控件等浏览器绘制部分 |
| outline | 轮廓简写 | 指示键盘焦点 |

@media、@container、@supports 是 @ 规则。minmax() 是值函数，:focus、:invalid 和 :hover 是伪类选择器，不属于属性。

## 3 Flexbox 导航与 Grid 页面

.site-header 和 nav 管理一维排列，flex-wrap: wrap 允许导航折行；没有通过 order 改变阅读次序。页面主体用 Grid 管理正文和侧栏。

52rem 是根据本例两列内容所需空间选择的断点，并不代表某一种设备。基础规则只排一列；媒体查询满足时才给出两个轨道。媒体查询中的 rem 依据浏览器初始字号，不是页面临时修改的根字号。

minmax(0, 2fr) 与 minmax(0, 1fr) 把剩余空间按 2∶1 分配，同时允许轨道缩小到 0。项目还设置 min-inline-size: 0，内容换行另由后面的规则负责；解除最小尺寸本身不会自动断开长字符串。

```html
<a class="skip" href="#main">跳到正文</a>
<header class="site-header">
  <strong>阅读工作坊</strong>
  <nav aria-label="主导航">
    <a href="index.html">内容页</a>
    <a href="catalog.html">目录页</a>
    <a href="form.html">表单页</a>
  </nav>
</header>
```

```css
.site-header, nav {
  display: flex;
  flex-wrap: wrap;
  align-items: center;
  gap: 0.75rem 1.5rem;
}
.site-header { justify-content: space-between; }
/* 缩窄视口：导航允许换行，DOM 顺序仍是内容、目录、表单。 */

.page-layout { display: grid; gap: 1.5rem; }
.catalog { display: grid; gap: 1rem; }
@media (min-width: 52rem) {
  .page-layout { grid-template-columns: minmax(0, 2fr) minmax(0, 1fr); }
  .catalog { grid-template-columns: repeat(2, minmax(0, 1fr)); }
  /* 达到断点才分两列；基础单列样式适用于更窄的视口。 */
}
```

配套文件：[index.html](scripts/22-responsive-page-practice/index.html)、[styles.css](scripts/22-responsive-page-practice/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/22-responsive-page-practice/index.html)

## 4 同一卡片适应不同容器

.card-slot 是卡片外层；命名为 card 的尺寸查询以它的内容区宽度为条件，作用于里面的 .card。命名容器需要位于被修改元素的祖先链上，不能用自身尺寸查询去改变自身。

卡片默认竖排，容器至少为 24rem 时变成图片与文字两列。正文侧栏可以保持竖排，目录中的卡片则可横排，即使两个页面的视口同样宽。这里的容器查询 rem 按根元素字号解析。

@supports 只判断浏览器是否接受指定声明，不能证明布局正确。容器查询未实现或增强规则被禁用时，仍使用完整的单列卡片；不能先隐藏内容再等待增强条件把它显示出来。

```html
<div class="card-slot">
  <article class="card">
    <img src="layout.svg" alt="两个内容区块的布局示意" width="480" height="240">
    <div><h2><a href="index.html">阅读与记录</a></h2><p>用同一卡片适应正文区和窄侧栏。</p></div>
  </article>
</div>
```

```css
.card { display: grid; gap: 1rem; }
.card img { max-inline-size: 100%; block-size: auto; }
@supports (container-type: inline-size) {
  .card-slot { container-type: inline-size; container-name: card; }
  @container card (min-width: 24rem) {
    .card { grid-template-columns: 7rem minmax(0, 1fr); align-items: center; }
  }
}
/* 查询 .card-slot 的内容区宽度，改变其后代 .card；不支持时卡片仍为单列。 */
```

配套文件：[catalog.html](scripts/22-responsive-page-practice/catalog.html)、[styles.css](scripts/22-responsive-page-practice/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/22-responsive-page-practice/catalog.html)

## 5 窄屏、长内容和文字缩放

连续标识可能没有正常断点，overflow-wrap: anywhere 在需要时提供额外断点，并参与最小内容尺寸计算。图片只限制最大宽度，block-size: auto 保留比例；本地 SVG 是演示资源，不包含其他章节依赖。

正文、卡片和按钮不写固定高度，以便文字放大后继续增高。不要用页面级 overflow-x: hidden 掩盖越界，它可能连焦点轮廓或内容一起裁掉。

分别检查约 320 CSS 像素宽的重排，以及浏览器文字放大到 200% 时的内容和操作。页面缩放与仅文字缩放不是同一种测试；单纯把浏览器窗口调窄不能代替二者。

```html
<p class="long-text">资料标识：ResponsiveLayoutReviewWithAnIntentionallyLongUnbrokenIdentifierForOverflowChecks2026</p>
<img class="article-image" src="layout.svg" alt="两个并排区块，上方是标题区" width="480" height="240">
```

```css
.long-text, .card, label { overflow-wrap: anywhere; }
.article-image { display: block; max-inline-size: 100%; block-size: auto; }
/* 长标识可断行；图片缩小而不拉伸，正文和表单都不固定高度。 */
```

配套文件：[index.html](scripts/22-responsive-page-practice/index.html)、[styles.css](scripts/22-responsive-page-practice/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/22-responsive-page-practice/index.html)

## 6 表单与键盘状态

&lt;label&gt; 通过 for 关联控件 id；required 和 type="email" 启用 HTML 原生约束校验。CSS 只负责外观，不能替代校验、保存数据或提供完整错误说明。

.field input 将文本输入框撑满可用宽度；复选框放在 .check 中，不被同一条宽度规则拉大。:focus:invalid 表示同时获得焦点且不满足约束，不会只因为页面刚打开就给所有必填项加错误边框。

:focus 匹配当前有焦点的元素，本例对鼠标或键盘产生的焦点都保留轮廓。链接同时保留下划线，必填条件同时用文字表达。按 Tab、Shift+Tab 和 Enter 检查顺序、焦点是否可见、提交校验能否操作。

表单仅重新请求本地 form.html，GET 参数会写入地址栏；使用虚构数据，不把它理解成已经保存的提交记录。

```html
<form action="form.html" method="get">
  <div class="field">
    <label for="email">邮箱（必填）</label>
    <input id="email" name="email" type="email" required aria-describedby="email-help">
    <p id="email-help">请输入含 @ 的邮箱；只用虚构数据体验校验。</p>
  </div>
  <div class="field">
    <label for="message">阅读建议</label>
    <textarea id="message" name="message" rows="4"></textarea>
  </div>
  <label class="check"><input type="checkbox" name="digest">希望收到摘要（演示选项）</label>
  <button type="submit">检查并重新打开本页</button>
  <!-- 仅演示原生校验；静态服务器不保存表单，GET 参数会出现在地址栏。 -->
</form>
```

```css
form { max-inline-size: 40rem; }
.field { margin-block: 1rem; }
.field label { display: block; }
.field input, textarea {
  inline-size: 100%;
  min-inline-size: 0;
  padding: 0.5em;
  border: 2px solid var(--line);
  background: var(--paper);
}
textarea { resize: vertical; }
.check { display: block; margin-block: 1rem; }
input:focus:invalid { border-color: var(--error); }
/* 必填提示用文字表达；浏览器校验消息仍由原生表单提供。 */

a:focus, button:focus, input:focus, textarea:focus {
  outline: 3px solid var(--link);
  outline-offset: 3px;
}
a:hover { text-decoration-thickness: 0.18em; }
/* 用 Tab 检查所有链接和控件；不删除原生操作，也不依赖 hover 才显示内容。 */
```

配套文件：[form.html](scripts/22-responsive-page-practice/form.html)、[styles.css](scripts/22-responsive-page-practice/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/22-responsive-page-practice/form.html)

## 7 明暗主题与兼容检查

prefers-color-scheme 是媒体特征；dark 是它的值。主题覆盖自定义属性后，三个页面共同改变。color-scheme 则告知浏览器本页采用的方案，影响表单控件等浏览器绘制部分；它不会自动替作者改好所有文字和背景。

以 WCAG 2.2 AA 为检查基准：普通文字与实际背景至少 4.5∶1；大文字可用 3∶1，大文字指至少 18pt，或至少 14pt 且粗体。识别控件或其状态所必需的视觉边界，应与相邻颜色达到 3∶1。不要把两个状态彼此之间的颜色差当作这个比值。

在目标浏览器中分别查看明暗主题、焦点、长内容和断点两侧；用颜色检查工具读取实际前景、背景，尤其注意有透明背景的元素。更换配色后需要重新检查，不能沿用旧结论。

兼容排查从样式资源加载开始，再检查规则是否匹配、容器类型是否被接受及增强后的实际尺寸。在开发者工具中关闭容器查询规则，确认基础卡片仍可读；本章示例没有据此承诺所有浏览器一致。

```html
<input id="email" name="email" type="email" required aria-describedby="email-help">
```

```css
@media (prefers-color-scheme: dark) {
  :root {
    --paper: #17212b;
    --ink: #f4f7fa;
    --link: #a9d2ff;
    --line: #bdc9d6;
    --error: #ffb4b4;
    color-scheme: dark;
  }
}
/* 切到 dark 后：背景 #17212b、文字 #f4f7fa、链接 #a9d2ff、边界 #bdc9d6。
   原生控件采用 dark 配色；其具体外观由浏览器绘制。 */
```

配套文件：[form.html](scripts/22-responsive-page-practice/form.html)、[styles.css](scripts/22-responsive-page-practice/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/22-responsive-page-practice/form.html)

## 本章小结

- 统一样式复用颜色、组件与状态；三种页面仍保留各自的语义结构。
- 媒体查询调整页面，容器查询调整卡片；两者观察的尺寸不同。
- 轨道可缩小、长文本可断行、图片可缩放，要共同配合。
- 响应式验收包括键盘、文字缩放、对比度及增强失效后的可用性。

## 练习

（1）把目录中一个标题增加到 60 个汉字，在 320px、831px、832px 和 1200px 视口检查三页。标准：文字完整，导航与表单可操作，页面无非预期横向滚动。

（2）只改变卡片外层的可用宽度，保持视口不变。标准：跨过 24rem 条件时，卡片在一列与两列间切换；禁用容器查询后保留可读单列。

（3）在浏览器中做 200% 文字放大和键盘遍历，再切换明暗主题测量正文、链接与控件边界对比度。标准：没有内容截断，焦点可见，颜色达到正文给出的适用阈值。

### 提示

先保留原始文件副本；用开发者工具检查容器内容区尺寸和计算样式。根字号改大只是额外压力测试，不能冒充浏览器真实缩放测试。

## 参考与引用来源

- MDN：[Responsive web design](https://developer.mozilla.org/en-US/docs/Learn_web_development/Core/CSS_layout/Responsive_Design) 的流式布局、断点、Flexbox/Grid、图片和 viewport；[Container queries](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Containment/Container_queries) 的尺寸上下文与回退；[minmax()](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Values/minmax)、[min-width 的 auto 值](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/min-width#values)、[overflow-wrap 的 anywhere 值](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/overflow-wrap#values) 的最小尺寸与断行；[UI pseudo-classes](https://developer.mozilla.org/en-US/docs/Learn_web_development/Extensions/Forms/UI_pseudo-classes) 的状态与约束校验；[:focus-visible 的焦点回退](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Selectors/:focus-visible#providing_a_focus_fallback)；[prefers-color-scheme](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/At-rules/@media/prefers-color-scheme)、[color-scheme](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/color-scheme) 的主题分工；[Using feature queries](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Conditional_rules/Using_feature_queries) 的声明支持检测。
- W3C：[Media Queries Level 4 §1.3](https://www.w3.org/TR/mediaqueries-4/#units) 与 [CSS Conditional Rules Level 5 §6.1](https://www.w3.org/TR/css-conditional-5/#size-container) 的媒体／容器查询相对长度参照；WAI 的 [Reflow](https://www.w3.org/WAI/WCAG22/Understanding/reflow.html) 说明 320 CSS 像素与缩放关系，[Resize Text](https://www.w3.org/WAI/WCAG22/Understanding/resize-text.html) 说明 200% 内容与功能检查；[Contrast (Minimum)](https://www.w3.org/WAI/WCAG22/Understanding/contrast-minimum.html) 与 [Non-text Contrast](https://www.w3.org/WAI/WCAG22/Understanding/non-text-contrast.html) 说明适用对象和阈值。
- Python 3.12：[http.server 命令行](https://docs.python.org/3.12/library/http.server.html#command-line-interface) 的工作目录、端口与绑定地址。